# 📊 PIP Engagement Analysis

This notebook analyzes the `analytic_logs` collection from Firestore to compute the **North Star Metrics** defined in our Analytics Strategy.

## Objectives
1. **Navigation & Stickiness**: Analyze screen stay times and user flows.
2. **Funnel Conversion**: Measure 'Write Flow' completion rates.
3. **Retention**: Calculate DAU/MAU stickiness.
4. **Quality**: Events per session analysis.

In [1]:
import firebase_admin
from firebase_admin import credentials, firestore
import pandas as pd
import seaborn as sns
import matplotlib.pyplot as plt
from datetime import datetime, timedelta

# 1. Initialize Firebase (Ensure serviceAccountKey.json is in this folder)
if not firebase_admin._apps:
    cred = credentials.Certificate("serviceAccountKey.json")
    firebase_admin.initialize_app(cred)

db = firestore.client()
print("🔥 Firebase Initialized")

🔥 Firebase Initialized


## 1. Load Data

In [4]:
def load_logs():
    logs_ref = db.collection('analytic_logs')
    docs = logs_ref.stream()
    
    data = []
    for doc in docs:
        d = doc.to_dict()
        if 'startTime' in d and d['startTime']:
             d['startTime'] = d['startTime'].replace(tzinfo=None) # naive for simplifiction
        if 'endTime' in d and d['endTime']:
             d['endTime'] = d['endTime'].replace(tzinfo=None)
        data.append(d)
    
    return pd.DataFrame(data)

df = load_logs()
print(f"📦 Loaded {len(df)} logs")
df.head()

📦 Loaded 3 logs


,startTime,subject_id,metrics,status,category,endTime,sessionType,id
0,2026-01-15 12:49:55.037894,622FACD0-FBA0-4BBD-8F95-5A4B7326F195,"{'total_duration': 36.393691062927246, 'steps'...",completed,write_view,2026-01-15 12:50:31.431586,write_view_daily,B368DC01-AD43-458F-93B3-578EAF8A5EC4
1,2026-01-15 12:53:03.813052,622FACD0-FBA0-4BBD-8F95-5A4B7326F195,"{'total_duration': 10.115086913108826, 'steps'...",completed,navigation,2026-01-15 12:53:13.928138,navigation_session,F7AFFAFB-398A-4C5E-B310-EC75C77BA415
2,2026-01-15 12:48:08.525861,622FACD0-FBA0-4BBD-8F95-5A4B7326F195,"{'total_duration': 103.33997297286987, 'step_e...",completed,onboarding,2026-01-15 12:49:51.865833,onboarding,FC4A42B0-E8ED-4CE5-BCC0-1ECE73BBB3DB


## 2. Navigation Analysis (Stickiness)

In [7]:
# Flatten 'metrics.steps' from navigation sessions
nav_logs = df[df['sessionType'] == 'navigation_session'].copy()

screen_views = []
for _, row in nav_logs.iterrows():
    metrics = row.get('metrics', {})
    steps = metrics.get('steps', [])
    if isinstance(steps, list):
        for step in steps:
            if isinstance(step, dict) and step.get('type') == 'screen_view':
                screen_views.append({
                    'session_id': row['id'],
                    'screen_name': step.get('screen_name'),
                    'timestamp': datetime.fromtimestamp(step.get('timestamp', 0))
                })

df_screens = pd.DataFrame(screen_views)

# Top Screns
if not df_screens.empty:
    plt.figure(figsize=(10, 5))
    sns.countplot(data=df_screens, y='screen_name', order=df_screens['screen_name'].value_counts().index)
    plt.title('Most Visited Screens')
    plt.show()

## 3. Funnel Conversion (Write Flow)

In [15]:
write_logs = df[df['sessionType'] == 'write_view_daily'].copy()

if not write_logs.empty:
    total_attempts = len(write_logs)
    completed = len(write_logs[write_logs['status'] == 'completed'])
    conversion_rate = (completed / total_attempts) * 100

    print(f"📝 Write Flow Conversion Rate: {conversion_rate:.2f}% ({completed}/{total_attempts})")

    # Completion Status Chart
    plt.figure(figsize=(6, 4))
    sns.countplot(data=write_logs, x='status')
    plt.title('Write Creation Outcomes')
    plt.show()
    
    # --- NEW: Duration Analysis per Card ---
    print("\n⏱️ Card Duration Analysis:")
    
    card_durations = []
    for _, row in write_logs.iterrows():
        metrics = row.get('metrics', {})
        steps = metrics.get('steps', [])
        
        if isinstance(steps, list):
            for step in steps:
                if isinstance(step, dict) and 'card_type' in step and 'duration_s' in step:
                    card_durations.append({
                        'card_type': step['card_type'],
                        'duration_s': step['duration_s']
                    })
    
    if card_durations:
        df_durations = pd.DataFrame(card_durations)
        
        # Plot Boxplot for Duration per Card Type
        plt.figure(figsize=(10, 6))
        sns.boxplot(data=df_durations, x='card_type', y='duration_s')
        plt.title('Time Spent per Card Type')
        plt.ylabel('Seconds')
        plt.xticks(rotation=45)
        plt.ylim(0, 60) # Limit y-axis to see distribution better (ignore outliers visually)
        plt.grid(True, axis='y', linestyle='--', alpha=0.7)
        plt.show()
        
        # Average Duration Table
        avg_durations = df_durations.groupby('card_type')['duration_s'].mean().sort_values(ascending=False)
        print(avg_durations)
    else:
        print("ℹ️ No detailed step duration data found yet.")

## 4. User Retention (DAU/MAU Stickiness)

In [11]:
df['date'] = pd.to_datetime(df['startTime']).dt.date

# Calculate DAU (Daily Active Users)
dau = df.groupby('date')['subject_id'].nunique()

# Calculate MAU (Monthly Active Users - approximations for demo)
df['month'] = pd.to_datetime(df['startTime']).dt.to_period('M')
mau = df.groupby('month')['subject_id'].nunique()

# Plot DAU
plt.figure(figsize=(12, 5))
dau.plot(kind='line', marker='o')
plt.title('Daily Active Users (DAU)')
plt.grid(True)
plt.show()